In [1]:
from spin_lattices import KagomeLattice, SpinLattice, ChainLattice, SquareLattice, TriangleLattice
from heisenberg_hamiltonians import HeisenbergJ1J2, SpinSystem
from pathlib import Path
import networkx as nx
import numpy as np
from typing import Callable
import torch
import numpy.typing as npt
import lattice_symmetries as ls
from typing import Any, Optional, Union, Dict, Tuple
from loguru import logger
from collections import namedtuple
from torch import Tensor
import torch.nn as nn
from misc_utils import make_unpacked_configurations
import io
from contextlib import redirect_stderr
from torch.utils.tensorboard import SummaryWriter
from datetime import datetime
from nqs_playground_helpers import (
    SamplingOptions,
    split_into_batches,
    safe_exp,
    sample_exactly,
    sample_full,
    forward_with_batches,
)
from scipy.sparse import csr_matrix, coo_matrix, diags
from scipy.sparse.csgraph import connected_components
import sys
from kagome_cnn import KagomeCNNRegression
from torch.nn.utils.convert_parameters import parameters_to_vector, vector_to_parameters
import time
from scipy.optimize import minimize
from vmc_amplitude import almost_true_relsigns
from vmc_vs_lbfgs import ScipyOptimizer
import fire
from my_stopwatch import stopwatch
from vmc_vs_lbfgs_2023_08_02 import LogProbDenseNet
from slater_determinant import SlaterDeterminant, Initializer

logger.remove()
logger.add(sys.stderr, level="INFO")

2023-08-04 20:02:02.406 | DEBUG    | lattice_symmetries:__init__:50 - Initializing Haskell runtime...
2023-08-04 20:02:02.416 | DEBUG    | lattice_symmetries:__init__:52 - Initializing Chapel runtime...
2023-08-04 20:02:02.522 | DEBUG    | lattice_symmetries:__init__:54 - Setting Python exception handler...
set_python_exception_handler ...


2

In [7]:
class AbsSlaterDet(nn.Module):
    def __init__(self, system: SpinSystem, initialization: str | Initializer = "orthogonal", postprocessor = None):
        super().__init__()
        self.det = SlaterDeterminant(
            lattice=system.lattice,
            basis=system.canonical_basis,
            initialization=initialization,
            sign_cache_dir=Path("signs_cache"),
        )
        self.system = system
        self.postprocessor = postprocessor

    def forward(self, states: torch.Tensor | npt.NDArray):
        if isinstance(states, np.ndarray):
            states = torch.from_numpy(states.astype(np.int64))
        indices = self.system.canonical_basis.index(states.detach().numpy().astype(np.uint64))
        assert isinstance(indices, np.ndarray)
        indices = torch.from_numpy(indices.astype(np.int64))
        ret = (torch.abs(self.det.forward(indices))).view(-1, 1)
        if self.postprocessor is not None:
            return self.postprocessor(ret)
        return ret


In [10]:
lattice = KagomeLattice(2, 3)
system = HeisenbergJ1J2(
    lattice=lattice,
    J1=1,
    J2=1,
    use_symmetries=False,
    spin_inversion=None,
    ground_state_cache_dir=Path("groundstates"),
)
logger.info(f"System: {system.get_cache_id()}")

energy, _ = system.get_eigenstates(1)
# log_prob_fn = LogProbDenseNet(system, n_hidden=1024, hidden_layers=3)

log_prob_fn = AbsSlaterDet(system)

logger.info(f"True energy: {energy[0]}")

writer = SummaryWriter(
    log_dir=(
        f"experiments/{datetime.now().strftime('%Y_%m_%d')}/{datetime.now().strftime('%H_%M_%S')}"
    )
)

optimizer = ScipyOptimizer(
    system=system,
    log_prob_fn=log_prob_fn,
    method="L-BFGS-B",
    maxiter=100000,
    batch_size=8096,
    tb_writer=writer,
)
try:
    r = optimizer.optimize()
    print(r)
finally:
    logger.info(str(stopwatch))


[Debug]   [LOCALE0]   ls_chpl_enumerate_representatives ...
2023-08-04 20:12:20.872 | INFO     | __main__:<module>:10 - System: HeisenbergJ1J2-KagomeLattice2x3-1.0-1.0-False-None
2023-08-04 20:12:20.928 | INFO     | __main__:<module>:17 - True energy: -32.19308309416496
/vol/tcm10/ischurov/frustrations-eda/vmc_amplitude.py:338: RuntimeWarning: divide by zero encountered in log
  np.log((nbd_matrix_w_signs @ abs_psi_nbd).astype(np.complex128))
2023-08-04 20:12:21.652 | INFO     | vmc_vs_lbfgs:objective:139 - -13.847854614257812
2023-08-04 20:12:23.001 | INFO     | vmc_vs_lbfgs:objective:139 - -13.847856521606445
2023-08-04 20:12:24.278 | INFO     | vmc_vs_lbfgs:objective:139 - -13.847891807556152
2023-08-04 20:12:25.550 | INFO     | vmc_vs_lbfgs:objective:139 - -13.853838920593262
2023-08-04 20:12:26.840 | INFO     | vmc_vs_lbfgs:objective:139 - nan
/vol/tcm10/ischurov/frustrations-eda/vmc_vs_lbfgs.py:145: RuntimeWarning: divide by zero encountered in log
  weighted_E_loc = torch.exp(lo

  message: CONVERGENCE: REL_REDUCTION_OF_F_<=_FACTR*EPSMCH
  success: True
   status: 0
      fun: -13.565177917480469
        x: [ 6.428e-01 -5.609e-01 ... -3.915e-01  6.117e-01]
      nit: 2
      jac: [ 0.000e+00 -2.767e+00 ... -2.378e+00  0.000e+00]
     nfev: 11
     njev: 11
 hess_inv: <324x324 LbfgsInvHessProduct with dtype=float64>
